# Notebook 2 — Decoding the Surface Code by Hand

*Part of **QEC Explorer**. Notebook 1 built the code and showed you how to *detect* errors. This one tackles the harder half: given only the lit detectors, **decide which correction to apply.** That's decoding — and unlike detection, there's no guaranteed-correct answer.*

You'll build **three real decoders** from scratch in plain Python — the same three that race in the Module 2 web tool:

1. **Lookup table** — precompute every syndrome's best correction (works only at small distance, but it's exact and a perfect baseline).
2. **Minimum-weight perfect matching (MWPM)** — the workhorse real decoder: turn the syndrome into a graph-matching problem and solve it.
3. **Belief propagation (BP)** — iterative message-passing, the kind of decoder that scales to large codes (and, as you'll see, has real weaknesses on small ones).

No PyMatching, no Stim — every algorithm is written out so you can step through it. As before, **assertion cells prove the Python reproduces the live tool's decoders exactly**, including their honest failures.

> ### 📺 Race them live
> Keep the **Module 2 — Decoder Race** tool open alongside this notebook:
> **→ [QEC Explorer · Module 2 (live)](https://github.com/kondshk/QEC-Explorer)**
> Every example here — especially the **Z(1,2)** case — is one you can replay there and watch animate.

**What stays out** (kept for later): noise models and thresholds (Notebook 3), and Qiskit/Stim circuit-level decoding. Here, decoding is pure classical inference over the syndrome.

---
## Setup — rebuild the surface code from Notebook 1

A decoder needs a code to decode. We re-paste the **exact** surface-code build from Notebook 1 (data qubits, checkerboard stabilizers, syndrome parity) so this notebook stands alone. If you've just run Notebook 1, this will look familiar — it's the same verified physics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

np.random.seed(0)
plt.rcParams["figure.dpi"] = 110

# ---- from Notebook 1: data qubits, stabilizers, syndrome (ports of lattice-core.js) ----
def build_data_qubits(d):
    return [(r, c) for r in range(d) for c in range(d)]

def build_stabilizers(d):
    stabs = []
    for R in range(-1, d):
        for C in range(-1, d):
            stype = "Z" if (R + C) % 2 == 0 else "X"
            corners = [(R, C), (R, C + 1), (R + 1, C), (R + 1, C + 1)]
            cand = [(rr, cc) for (rr, cc) in corners if 0 <= rr < d and 0 <= cc < d]
            if len(cand) == 0:
                continue
            on_top_bottom = (R == -1 or R == d - 1)
            on_left_right = (C == -1 or C == d - 1)
            if len(cand) == 2:
                if stype == "Z" and not on_left_right: continue
                if stype == "X" and not on_top_bottom: continue
            if len(cand) not in (2, 4):
                continue
            stabs.append({"type": stype, "center": (C + 0.5, R + 0.5), "data": cand})
    return stabs

def compute_syndrome(stabs, errors):
    syn = []
    for s in stabs:
        parity = 0
        for (r, c) in s["data"]:
            e = errors.get((r, c))
            if not e: continue
            if s["type"] == "Z" and e.get("x"): parity ^= 1
            if s["type"] == "X" and e.get("z"): parity ^= 1
        syn.append(parity)
    return syn

d = 3
data = build_data_qubits(d)
stabs = build_stabilizers(d)
print(f"d={d}: {len(data)} data qubits, {len(stabs)} stabilizers — code ready to decode.")

We also need two small **Pauli-algebra helpers** the decoders rely on. An error/correction is a dict `{(r,c): {"x":bool,"z":bool}}`. Applying a correction means **XOR-ing** its flips onto the error (these are ports of `combineErrors` / `errorWeight` from `lattice-core.js`):

In [ ]:
def combine_errors(a, b):
    # XOR two error sets per Pauli component. A qubit ending in identity is dropped.
    out = {}
    for k in set(a) | set(b):
        ea = a.get(k, {"x": False, "z": False})
        eb = b.get(k, {"x": False, "z": False})
        x = bool(ea.get("x")) ^ bool(eb.get("x"))
        z = bool(ea.get("z")) ^ bool(eb.get("z"))
        if x or z:
            out[k] = {"x": x, "z": z}
    return out

def error_weight(errors):
    # number of non-identity qubits (Y counts as 1 — it's one physical qubit)
    return sum(1 for k in errors if errors[k].get("x") or errors[k].get("z"))

def diff_keys(a, b):
    # qubits where two corrections differ in either component (for degeneracy checks)
    out = []
    for k in set(a) | set(b):
        ea = a.get(k, {"x": False, "z": False}); eb = b.get(k, {"x": False, "z": False})
        if bool(ea.get("x")) != bool(eb.get("x")) or bool(ea.get("z")) != bool(eb.get("z")):
            out.append(k)
    return sorted(out)

# quick sanity: applying an error to itself cancels to identity
e = {(1,1): {"x": True, "z": False}}
print("error XOR itself =", combine_errors(e, e), " (empty = cancelled ✓)")

---
## 1 · What decoding actually is (and why it's hard)

Detection gave us a **syndrome** — the list of lit detectors. A decoder's job: **propose a correction** (a set of Pauli flips) that, applied to the data, cancels the syndrome and — hopefully — undoes the real error.

Here's the catch you discovered at the end of Notebook 1. **Many different errors produce the same syndrome.** The decoder never sees the error; it sees only the syndrome, and must *guess* which error happened. The standard principle: pick the **most likely** explanation, which for low error rates means the **lowest-weight** error consistent with the syndrome.

But "lowest weight" isn't always unique, and even when the decoder guesses a valid low-weight correction, it can **return the code to safety while silently completing a logical operator** — a correct-looking fix that actually corrupts the data. We'll watch all three decoders fall into exactly that trap on the **Z(1,2)** error.

### The success criterion: residual, not match

A decoder does **not** need to guess the exact error. It needs its correction's **residual** — `error XOR correction` — to be a *product of stabilizers* (silent syndrome, no logical flip). Three outcomes, ported from `evaluateCorrection`:

| Residual | Verdict |
|---|---|
| silent syndrome **and** no logical flip | ✅ **Fixed** |
| silent syndrome **but** a logical operator | ❌ **Logical error introduced** (looks fixed, isn't) |
| syndrome still lit | ❌ **Left codespace** (didn't even cancel the syndrome) |

In [ ]:
# We need logical_status from Notebook 1 to judge residuals. (Port of logicalStatus.)
def logical_status(stabs, errors, d):
    syn = compute_syndrome(stabs, errors)
    if any(syn):
        return {"logical": False, "detectable": True}
    x_col0 = z_row0 = 0
    for r in range(d):
        e = errors.get((r, 0))
        if e and e.get("x"): x_col0 ^= 1
    for c in range(d):
        e = errors.get((0, c))
        if e and e.get("z"): z_row0 ^= 1
    return {"logical": (x_col0 == 1 or z_row0 == 1), "detectable": False}

def evaluate_correction(stabs, original_errors, correction, d):
    # Port of evaluateCorrection(). Returns status in {fixed, logical-introduced, left-codespace}.
    residual = combine_errors(original_errors, correction)
    if any(compute_syndrome(stabs, residual)):
        return {"status": "left-codespace", "ok": False, "label": "Did not return to codespace ✗"}
    if logical_status(stabs, residual, d)["logical"]:
        return {"status": "logical-introduced", "ok": False, "label": "Logical error introduced ✗"}
    return {"status": "fixed", "ok": True, "label": "Fixed ✓"}

# sanity: the perfect correction (== the error) always fixes a detectable error
err = {(1,1): {"x": True, "z": False}}
print("perfect correction verdict:", evaluate_correction(stabs, err, err, d)["label"])

---
## 2 · One trick that simplifies everything: split X and Z

A key simplification every one of these decoders uses: **X errors and Z errors don't interfere.**

- **X errors** (bit-flips) are detected *only* by **Z-type** stabilizers.
- **Z errors** (phase-flips) are detected *only* by **X-type** stabilizers.

So we can decode the two **channels independently** — solve the X-error problem using the Z-detectors, solve the Z-error problem using the X-detectors, then merge. A `Y` error is just a qubit that's in *both* the X channel and the Z channel. This is why the code below always has a `channel` parameter.

In [ ]:
DETECTOR_TYPE = {"X": "Z", "Z": "X"}   # channel -> the stabilizer type that detects it

def stabs_of_type(stabs, stype):
    # return [(index, stabilizer), ...] for stabilizers of one type
    return [(i, s) for i, s in enumerate(stabs) if s["type"] == stype]

def apply_flips(correction, qubit_keys, channel):
    # XOR a channel-flip onto each listed qubit (mutates `correction`)
    for k in qubit_keys:
        cur = correction.get(k, {"x": False, "z": False}).copy()
        if channel == "X": cur["x"] = not cur["x"]
        else:              cur["z"] = not cur["z"]
        if not cur["x"] and not cur["z"]: correction.pop(k, None)
        else:                              correction[k] = cur

print("X channel is read by", DETECTOR_TYPE["X"], "-type stabilizers;",
      "Z channel by", DETECTOR_TYPE["Z"], "-type.")

---
## 3 · Decoder #1 — the lookup table (exact, but it doesn't scale)

The simplest possible decoder: **precompute the answer for every syndrome.** Enumerate all low-weight errors, compute each one's syndrome, and remember the lowest-weight correction for each syndrome you see. Decoding is then a dictionary lookup.

This is **exact** (it genuinely finds a minimum-weight correction) and makes a perfect baseline — but the table has $2^{d^2-1}$ possible syndromes, so it's only practical at **d = 3**. We build it by enumerating all weight-≤2 errors, a direct port of `buildLookupTable`.

**The honest bit:** when two *different* corrections of the *same* weight explain one syndrome, the table has to pick one — but it records that the syndrome is **degenerate**, because the decoder genuinely can't know which error really happened. Watch for that flag.

In [ ]:
def build_lookup_table(stabs, data, d):
    # Port of buildLookupTable(): d=3 only. Enumerate weight 0,1,2 errors.
    if d != 3:
        return None
    table = {}   # syndrome-tuple -> {"weight", "correction", "degenerate"}
    keys = [f"{r},{c}" for (r, c) in data]   # for printing; we key by (r,c) below

    def syn_key(errors):
        return tuple(compute_syndrome(stabs, errors))

    def consider(errors):
        k = syn_key(errors)
        w = error_weight(errors)
        prev = table.get(k)
        if prev is None or w < prev["weight"]:
            table[k] = {"weight": w, "correction": dict(errors), "degenerate": False}
        elif w == prev["weight"] and len(diff_keys(prev["correction"], errors)) > 0:
            prev["degenerate"] = True   # a genuinely different equal-weight correction exists

    paulis = [{"x": True, "z": False}, {"x": False, "z": True}, {"x": True, "z": True}]
    consider({})                                            # weight 0
    for q in data:                                          # weight 1
        for p in paulis:
            consider({q: dict(p)})
    for i in range(len(data)):                              # weight 2
        for j in range(i + 1, len(data)):
            for pa in paulis:
                for pb in paulis:
                    consider({data[i]: dict(pa), data[j]: dict(pb)})
    return table

def lookup_decode(stabs, table, errors):
    # Port of lookupDecode(). Returns {correction, note, degenerate}.
    if table is None:
        return {"available": False, "correction": {}, "degenerate": False,
                "note": "Lookup only practical at d=3 — table grows exponentially with distance."}
    hit = table.get(tuple(compute_syndrome(stabs, errors)))
    if hit is None:
        return {"available": True, "correction": {}, "degenerate": False,
                "note": "Syndrome not in the weight≤2 table (error too heavy)."}
    note = ("one of several equally likely corrections (degeneracy — can't tell which is right "
            "from the syndrome alone)") if hit["degenerate"] else "minimal-weight correction"
    return {"available": True, "correction": dict(hit["correction"]),
            "degenerate": hit["degenerate"], "note": "Looked up syndrome → " + note + "."}

lookup_table = build_lookup_table(stabs, data, d)
print(f"Lookup table built: {len(lookup_table)} distinct syndromes mapped.\n")

for name, err in [("single X(1,1)", {(1,1): {"x": True, "z": False}}),
                  ("single Z(1,1)", {(1,1): {"x": False, "z": True}})]:
    res = lookup_decode(stabs, lookup_table, err)
    verdict = evaluate_correction(stabs, err, res["correction"], d)
    print(f"{name}: correction={res['correction']}  ->  {verdict['label']}")

---
## 4 · Decoder #2 — minimum-weight perfect matching (the real workhorse)

The lookup table doesn't scale, so real surface-code decoders almost always start from **MWPM**. The insight is beautiful:

> A single error chain in one channel lights up the **two detectors at its endpoints** (or one detector + the boundary, for a chain ending at an edge). So the lit detectors come in pairs that want to be **connected back up** — and the most-likely error is the one connecting them along the **shortest paths**.

So decoding becomes a graph problem:
- **Nodes** = the detectors of this channel, plus one virtual **BOUNDARY** node.
- **Edges** = data qubits. Flipping a qubit toggles the (one or two) detectors it touches — so each qubit is an edge between two nodes (or a node and the boundary).
- A lit detector is a **defect**. We find the **minimum-weight perfect matching**: pair up all defects (a defect may pair to the boundary) so the total path length is smallest. The matched paths *are* the correction.

This is exactly what PyMatching does at scale. We build the graph, BFS for shortest paths, and solve the matching by exact enumeration (fine for the few defects at d=3). Ports of `buildChannelGraph`, `bfsFrom`, `exactMatch`, `mwpmChannel`.

In [ ]:
BOUNDARY = "BOUNDARY"

def build_channel_graph(stabs, data, channel):
    # Port of buildChannelGraph(): node = detector index (or BOUNDARY); edge = data qubit.
    det_type = DETECTOR_TYPE[channel]
    dets = stabs_of_type(stabs, det_type)            # [(idx, stab)]
    det_idx = [i for (i, s) in dets]
    adj = {BOUNDARY: []}
    for i in det_idx:
        adj[i] = []
    for (r, c) in data:
        # which detectors of this type touch this qubit?
        touch = [i for (i, s) in dets if (r, c) in s["data"]]
        if len(touch) == 2:
            adj[touch[0]].append((touch[1], (r, c)))
            adj[touch[1]].append((touch[0], (r, c)))
        elif len(touch) == 1:
            adj[touch[0]].append((BOUNDARY, (r, c)))
            adj[BOUNDARY].append((touch[0], (r, c)))
        # touches 0 detectors of this type -> no edge in this channel
    return adj

def bfs_from(adj, src):
    # Port of bfsFrom(): shortest path in #qubits from src to every node.
    dist = {src: 0}; prev_q = {}; prev_n = {}
    queue = [src]; head = 0
    while head < len(queue):
        u = queue[head]; head += 1
        for (to, qubit) in adj.get(u, []):
            if to not in dist:
                dist[to] = dist[u] + 1
                prev_q[to] = qubit; prev_n[to] = u
                queue.append(to)
    return {"dist": dist, "prev_q": prev_q, "prev_n": prev_n}

def reconstruct_qubits(bfs, src, target):
    # walk prev pointers from target back to src, collecting edge qubits
    qubits = []; cur = target; guard = 0
    while cur != src and cur in bfs["prev_n"] and guard < 10000:
        qubits.append(bfs["prev_q"][cur]); cur = bfs["prev_n"][cur]; guard += 1
    return qubits

def exact_match(n, pair_weight, boundary_weight):
    # Port of exactMatch(): min-weight perfect matching by enumeration.
    # Each defect pairs with a later defect or with the boundary.
    best = {"weight": None, "pairs": None}
    def recurse(used, acc, pairs):
        i = next((k for k in range(n) if not used[k]), -1)
        if i == -1:
            if best["weight"] is None or acc < best["weight"]:
                best["weight"] = acc; best["pairs"] = list(pairs)
            return
        if best["weight"] is not None and acc >= best["weight"]:
            return
        used[i] = True                                    # option A: i -> boundary
        recurse(used, acc + boundary_weight(i), pairs + [(i, -1)])
        used[i] = False
        for j in range(i + 1, n):                         # option B: i -> j
            if used[j]: continue
            used[i] = used[j] = True
            recurse(used, acc + pair_weight(i, j), pairs + [(i, j)])
            used[i] = used[j] = False
    recurse([False] * n, 0, [])
    return best if best["pairs"] is not None else {"weight": 0, "pairs": []}

In [ ]:
def mwpm_channel(stabs, data, errors, channel):
    # Port of mwpmChannel(): match defects, reconstruct correction along shortest paths.
    det_type = DETECTOR_TYPE[channel]
    syn = compute_syndrome(stabs, errors)
    dets = [i for (i, s) in stabs_of_type(stabs, det_type) if syn[i] == 1]
    result = {"correction": {}, "paths": []}
    if not dets:
        return result
    adj = build_channel_graph(stabs, data, channel)
    bfs_cache = {}
    def bfs_of(node):
        if node not in bfs_cache: bfs_cache[node] = bfs_from(adj, node)
        return bfs_cache[node]
    def dist_node(a, b):
        d_ = bfs_of(a)["dist"].get(b)
        return float("inf") if d_ is None else d_
    pw = lambda i, j: dist_node(dets[i], dets[j])
    bw = lambda i:    dist_node(dets[i], BOUNDARY)
    match = exact_match(len(dets), pw, bw)
    for (i, j) in match["pairs"]:
        if j == -1: qkeys = reconstruct_qubits(bfs_of(dets[i]), dets[i], BOUNDARY)
        else:       qkeys = reconstruct_qubits(bfs_of(dets[i]), dets[i], dets[j])
        apply_flips(result["correction"], qkeys, channel)
        result["paths"].append({"from": dets[i], "to": (None if j == -1 else dets[j]),
                                "qubits": qkeys, "channel": channel})
    return result

def mwpm_decode(stabs, data, errors):
    # Port of mwpmDecode(): decode both channels and merge.
    x = mwpm_channel(stabs, data, errors, "X")
    z = mwpm_channel(stabs, data, errors, "Z")
    return {"correction": combine_errors(x["correction"], z["correction"]),
            "paths": x["paths"] + z["paths"], "note": "Exact minimum-weight perfect matching."}

for name, err in [("single X(1,1)", {(1,1): {"x": True, "z": False}}),
                  ("two X (0,0)+(0,1)", {(0,0): {"x": True}, (0,1): {"x": True}})]:
    res = mwpm_decode(stabs, data, err)
    verdict = evaluate_correction(stabs, err, res["correction"], d)
    print(f"{name}: correction={res['correction']}  ->  {verdict['label']}")

---
## 5 · Decoder #3 — belief propagation (scales up, but stumbles here)

MWPM is great for the surface code, but it doesn't generalize to every code (it relies on each error lighting *exactly two* detectors). **Belief propagation** is the decoder that scales to large, general codes — it's iterative *message passing* on the **Tanner graph** (data qubits ↔ the stabilizers that check them).

Working in the **log-likelihood ratio (LLR)** domain, each qubit starts with a prior belief that it's probably fine (we assume physical error rate $p = 0.05$), then qubits and checks exchange messages:
- **Check → qubit:** "given my syndrome bit and what your neighbors tell me, here's how much I think *you* are flipped" (the `tanh` sum-product rule).
- **Qubit → check:** "here's my updated belief, excluding what you just told me."

After a few rounds the beliefs settle; any qubit whose final $P(\text{error}) > 0.5$ gets flipped. Direct port of `bpChannel` / `bpDecode`.

> **Spoiler, and an honest one:** plain BP is known to **struggle on the surface code** because the code is highly *degenerate* (many equivalent errors), and symmetric beliefs can deadlock. We will *not* fake its success. On some of the very cases MWPM nails, BP will leave the codespace. That's a real, published limitation — and exactly why research decoders bolt extra machinery (ordered-statistics post-processing, neural networks — including the kind of GNN work this project connects to) onto BP.

In [ ]:
BP_CHANNEL_P   = 0.05
BP_MAX_ITERS   = 20
BP_CONVERGE_EPS = 1e-3

def _clamp(x): return max(-30.0, min(30.0, x))

def bp_channel(stabs, data, errors, channel, trace=False):
    # Port of bpChannel(): LLR sum-product message passing on the Tanner graph.
    det_type = DETECTOR_TYPE[channel]
    checks = stabs_of_type(stabs, det_type)              # [(idx, stab)]
    syn = compute_syndrome(stabs, errors)

    var_keys = list(data)
    var_index = {k: i for i, k in enumerate(var_keys)}
    check_vars = [[var_index[k] for k in s["data"]] for (_, s) in checks]
    check_syn  = [syn[i] for (i, _) in checks]
    var_checks = [[] for _ in var_keys]
    for ci, vs in enumerate(check_vars):
        for vi in vs: var_checks[vi].append(ci)

    L0 = np.log((1 - BP_CHANNEL_P) / BP_CHANNEL_P)       # prior LLR (favors "no error")
    Lvc = [{vi: L0 for vi in vs} for vs in check_vars]   # var->check messages
    Lcv = [{vi: 0.0 for vi in vs} for vs in check_vars]  # check->var messages

    iters, converged, iter_marg = 0, False, []
    for it in range(BP_MAX_ITERS):
        iters = it + 1
        max_change = 0.0
        # check -> variable (tanh / sum-product rule, syndrome bit as sign)
        for ci, vs in enumerate(check_vars):
            sign = -1.0 if check_syn[ci] else 1.0
            for vi in vs:
                prod = 1.0
                for vj in vs:
                    if vj == vi: continue
                    prod *= np.tanh(_clamp(Lvc[ci][vj]) / 2.0)
                prod = max(-0.999999, min(0.999999, prod))
                msg = sign * 2.0 * np.arctanh(prod)
                max_change = max(max_change, abs(msg - Lcv[ci][vi]))
                Lcv[ci][vi] = msg
        # variable -> check
        for vi in range(len(var_keys)):
            cs = var_checks[vi]
            total = sum(Lcv[ci][vi] for ci in cs)
            for ci in cs:
                Lvc[ci][vi] = _clamp(L0 + (total - Lcv[ci][vi]))
        # per-iteration marginals  P(error) = sigmoid(-Lmarg)
        marg = [1.0 / (1.0 + np.exp(_clamp(L0 + sum(Lcv[ci][vi] for ci in var_checks[vi]))))
                for vi in range(len(var_keys))]
        if trace: iter_marg.append(marg)
        if max_change < BP_CONVERGE_EPS:
            converged = True; break

    correction = {}
    final_marg = [1.0 / (1.0 + np.exp(_clamp(L0 + sum(Lcv[ci][vi] for ci in var_checks[vi]))))
                  for vi in range(len(var_keys))]
    for vi, k in enumerate(var_keys):
        if final_marg[vi] > 0.5: apply_flips(correction, [k], channel)
    return {"correction": correction, "iters": iters, "converged": converged,
            "final_marg": final_marg, "iter_marg": iter_marg, "var_keys": var_keys}

def bp_decode(stabs, data, errors, trace=False):
    # Port of bpDecode(): both channels, merge.
    x = bp_channel(stabs, data, errors, "X", trace)
    z = bp_channel(stabs, data, errors, "Z", trace)
    iters = max(x["iters"], z["iters"])
    converged = x["converged"] and z["converged"]
    note = (f"Converged after {iters} iteration{'s' if iters != 1 else ''}."
            if converged else f"Did not converge after {BP_MAX_ITERS} iterations.")
    return {"correction": combine_errors(x["correction"], z["correction"]),
            "iters": iters, "converged": converged, "note": note, "channels": {"X": x, "Z": z}}

res = bp_decode(stabs, data, {(1,1): {"x": True, "z": False}}, trace=True)
print("BP on single X(1,1):", res["note"])
print("correction:", res["correction"], " ->",
      evaluate_correction(stabs, {(1,1): {"x": True}}, res["correction"], d)["label"])

---
## 6 · The race — all three on the same syndrome

Now run all three decoders on the same error and compare, exactly like the Module 2 tool. Let's start with cases where they agree, then find where they don't.

In [ ]:
def race(err, label):
    print(f"=== {label}   error = {pretty(err)} ===")
    syn = compute_syndrome(stabs, err)
    print(f"    syndrome: {''.join(map(str, syn))}")
    for name, res in [("lookup", lookup_decode(stabs, lookup_table, err)),
                      ("mwpm  ", mwpm_decode(stabs, data, err)),
                      ("bp    ", bp_decode(stabs, data, err))]:
        v = evaluate_correction(stabs, err, res["correction"], d)
        extra = ""
        if name == "lookup" and res.get("degenerate"): extra = "  [degenerate]"
        print(f"    {name}: corr={pretty(res['correction']):<16} {v['label']}{extra}")
    print()

def pretty(errors):
    if not errors: return "(none)"
    parts = []
    for k in sorted(errors):
        e = errors[k]; tag = ("X" if e.get("x") else "") + ("Z" if e.get("z") else "")
        parts.append(f"{k}:{tag}")
    return " ".join(parts)

race({(1,1): {"x": True, "z": False}}, "Single X error")
race({(1,1): {"x": True, "z": True}},  "Single Y error")
race({(0,0): {"x": True}, (2,2): {"z": True}}, "Two errors, separate channels")

On clean single errors all three usually agree and fix it. Notice already, in the **two-channel** case, that **BP can lag** — it may fix one channel but not both. Now the case that matters most.

### 6.1 · The Z(1,2) degeneracy — where "fixing the syndrome" betrays you

This is the exact error you discovered in Notebook 1: a single **Z at (1,2)** whose syndrome is shared by **Z at (0,2)**. Watch what each decoder does with that ambiguity.

In [ ]:
race({(1,2): {"x": False, "z": True}}, "Single Z(1,2)  — the degenerate boundary case")

Look carefully at that result — it's the whole point of the notebook:

- **Lookup** flags the syndrome **degenerate** and picks `Z(0,2)`. That correction *does* silence the syndrome — but `Z(1,2) XOR Z(0,2)` is a vertical Z-chain that, combined with the boundary, completes a **logical operator**. Verdict: **Logical error introduced.** The decoder was *reasonable* and still *wrong*.
- **MWPM** independently makes the *same* defensible choice (the defect is equidistant from two boundary routes) and lands in the same trap.
- **BP** can't even settle the symmetric beliefs and **leaves the codespace** entirely.

Not one of the three "wins." That's not a bug in our code — **it's the irreducible ambiguity of decoding.** A single low-weight error, perfectly legal syndrome, and the most reasonable corrections still corrupt the logical qubit. This is precisely why bigger code distance (Notebook 3's threshold story) is the only real defense — and why better decoders are an open research problem.

> ▶️ **Replay this in [Module 2 (live)](https://github.com/kondshk/QEC-Explorer)**: inject Z at (1,2), hit *Run race*, and watch the three panels reach exactly these verdicts in real time.

### 6.2 · The silent logical — a full column

And the cleanest illustration of "a fixed syndrome is not a fixed qubit": the full column-0 X error from Notebook 1. Its syndrome is **all zeros**, so every decoder correctly proposes **no correction** — and a logical error sails straight through, undetected.

In [ ]:
race({(0,0): {"x": True}, (1,0): {"x": True}, (2,0): {"x": True}}, "Full column-0 X  — silent logical")

---
## 7 · Seeing MWPM match the defects

To make the matching concrete, here's the lattice with the **lit defects** and the **shortest-path correction chains** MWPM drew between them — the same picture the Module 2 panel animates.

In [ ]:
def plot_mwpm(stabs, data, errors, d, title="MWPM matching"):
    res = mwpm_decode(stabs, data, errors)
    syn = compute_syndrome(stabs, errors)
    fig, ax = plt.subplots(figsize=(5.0, 5.0))
    Z_FILL, X_FILL = "#2ec4a0", "#e45d9a"

    for i, s in enumerate(stabs):
        pts = [(c, r) for (r, c) in s["data"]]; cx, cy = s["center"]
        pts.sort(key=lambda p: np.arctan2(p[1]-cy, p[0]-cx))
        lit = syn[i] == 1
        fill = "#9b6dff" if lit else (Z_FILL if s["type"] == "Z" else X_FILL)
        ax.add_patch(plt.Polygon(pts, closed=True, facecolor=fill,
                                 alpha=0.55 if lit else 0.16, edgecolor=fill, lw=1.3, zorder=1))

    for (r, c) in data:
        if (r, c+1) in data: ax.plot([c, c+1], [r, r], color="0.85", lw=1, zorder=1)
        if (r+1, c) in data: ax.plot([c, c], [r, r+1], color="0.85", lw=1, zorder=1)

    # correction chains
    for path in res["paths"]:
        col = "#2ec4a0" if path["channel"] == "X" else "#4a8fe7"
        for (r, c) in path["qubits"]:
            ax.add_patch(plt.Circle((c, r), 0.22, fill=False, ec=col, lw=2.5,
                                    ls=(0, (3, 2)), zorder=4))

    for (r, c) in data:
        e = errors.get((r, c))
        face = "#2a2a3a"
        if e and e.get("x") and e.get("z"): face = "#d4a855"
        elif e and e.get("x"):              face = "#ef6e4e"
        elif e and e.get("z"):              face = "#4a8fe7"
        ax.plot(c, r, "o", ms=15, color=face, mec="#43435f", mew=1.4, zorder=3)

    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_title(f"{title}\n(purple = lit defect, dashed ring = correction)", fontsize=10)
    ax.set_xlabel("column c →"); ax.set_ylabel("← row r")
    ax.set_xlim(-1.3, d-0.3); ax.set_ylim(d+0.6, -1.0)
    plt.tight_layout(); plt.show()

plot_mwpm(stabs, data, {(1,1): {"x": True}}, d, "MWPM — single X(1,1) (fixed)")
plot_mwpm(stabs, data, {(1,2): {"z": True}}, d, "MWPM — single Z(1,2) (degenerate trap)")

---
## 8 · Proof: these decoders match the live tool

The reference values below were generated by the **actual JavaScript decoders in `decoders.js`** (the engine behind Module 2). The assertions check that our hand-built Python reproduces them **exactly** — same corrections, same verdicts, same degeneracy flags, same BP convergence, *including the honest failure cases*. All green ✅ means your from-scratch decoders are faithful to the live tool.

In [ ]:
def check(name, condition):
    print(f"  {'✅' if condition else '❌'}  {name}")
    assert condition, f"MISMATCH: {name}"

def sig(corr):
    if not corr: return "(empty)"
    return "|".join(f"{r},{c}:{('X' if corr[(r,c)].get('x') else '')}{('Z' if corr[(r,c)].get('z') else '')}"
                    for (r, c) in sorted(corr))

print("Reproducing decoders.js reference outputs from the hand-built Python:\n")

# (status, lookup_sig, lookup_degenerate, mwpm_sig, mwpm_status, bp_sig, bp_status)
# Reference values captured directly from decoders.js (Module 2 engine).
cases = {
    "single X(1,1)": (
        {(1,1): {"x": True}},
        dict(lk="1,1:X", lk_deg=False, lk_st="fixed",
             mw="1,1:X", mw_st="fixed",
             bp="1,1:X", bp_st="fixed", bp_iters=5)),
    "single Z(1,1)": (
        {(1,1): {"z": True}},
        dict(lk="1,1:Z", lk_deg=False, lk_st="fixed",
             mw="1,1:Z", mw_st="fixed",
             bp="1,1:Z", bp_st="fixed", bp_iters=5)),
    "single Y(1,1)": (
        {(1,1): {"x": True, "z": True}},
        dict(lk="1,1:XZ", lk_deg=False, lk_st="fixed",
             mw="1,1:XZ", mw_st="fixed",
             bp="1,1:XZ", bp_st="fixed", bp_iters=5)),
    "single Z(1,2) [degenerate]": (
        {(1,2): {"z": True}},
        dict(lk="0,2:Z", lk_deg=True, lk_st="logical-introduced",
             mw="0,2:Z", mw_st="logical-introduced",
             bp="(empty)", bp_st="left-codespace", bp_iters=5)),
    "zero error": (
        {},
        dict(lk="(empty)", lk_deg=False, lk_st="fixed",
             mw="(empty)", mw_st="fixed",
             bp="(empty)", bp_st="fixed", bp_iters=5)),
    "full column-0 X [logical]": (
        {(0,0): {"x": True}, (1,0): {"x": True}, (2,0): {"x": True}},
        dict(lk="(empty)", lk_deg=False, lk_st="logical-introduced",
             mw="(empty)", mw_st="logical-introduced",
             bp="(empty)", bp_st="logical-introduced", bp_iters=5)),
    "X(0,0)+Z(2,2)": (
        {(0,0): {"x": True}, (2,2): {"z": True}},
        dict(lk="0,0:X|2,2:Z", lk_deg=True, lk_st="fixed",
             mw="0,0:X|2,2:Z", mw_st="fixed",
             bp="2,2:Z", bp_st="left-codespace", bp_iters=5)),
}

for name, (err, ref) in cases.items():
    lk = lookup_decode(stabs, lookup_table, err)
    mw = mwpm_decode(stabs, data, err)
    bp = bp_decode(stabs, data, err)
    lk_v = evaluate_correction(stabs, err, lk["correction"], d)
    mw_v = evaluate_correction(stabs, err, mw["correction"], d)
    bp_v = evaluate_correction(stabs, err, bp["correction"], d)
    ok = (sig(lk["correction"]) == ref["lk"] and bool(lk["degenerate"]) == ref["lk_deg"]
          and lk_v["status"] == ref["lk_st"]
          and sig(mw["correction"]) == ref["mw"] and mw_v["status"] == ref["mw_st"]
          and sig(bp["correction"]) == ref["bp"] and bp_v["status"] == ref["bp_st"]
          and bp["iters"] == ref["bp_iters"])
    check(f"{name}: lookup/MWPM/BP all match decoders.js", ok)

# structural guarantees that don't depend on the captured table
print()
# MWPM returns a minimum-weight (weight-1) correction for EVERY weight-1 error
min_ok = True
for (r, c) in data:
    for comp in ({"x": True}, {"z": True}):
        mw = mwpm_decode(stabs, data, {(r, c): comp})
        if error_weight(mw["correction"]) != 1:
            min_ok = False
check("MWPM returns a weight-1 (minimum) correction for every weight-1 error", min_ok)

# every weight-1 error that ISN'T boundary-degenerate is genuinely fixed by MWPM
fixed_or_degenerate = True
for (r, c) in data:
    for comp in ({"x": True}, {"z": True}):
        err = {(r, c): comp}
        v = evaluate_correction(stabs, err, mwpm_decode(stabs, data, err)["correction"], d)
        if v["status"] == "left-codespace":   # the one thing that must never happen
            fixed_or_degenerate = False
check("MWPM never leaves the codespace on a weight-1 error (worst case = boundary degeneracy)",
      fixed_or_degenerate)

# the L0 prior constant matches decoders.js exactly
check("BP prior L0 = log((1-0.05)/0.05) matches decoders.js",
      abs(np.log((1 - 0.05) / 0.05) - 2.9444389791664403) < 1e-9)

print("\n🎉  All three decoders agree with the live QEC Explorer engine — successes AND honest failures.")

---
## 9 · Wrap-up — and the bridge to Notebook 3

You built three real decoders from nothing — a lookup table, minimum-weight matching, and belief propagation — and proved them faithful to the live decoder-race tool, *including where they fail.* The big lessons:

- **Decoding is inference, not arithmetic.** The syndrome under-determines the error; the decoder must guess, and "reasonable" guesses can still introduce a logical error (the **Z(1,2)** trap).
- **Different decoders, different trade-offs.** Lookup is exact but doesn't scale; MWPM is the surface-code workhorse but assumes two-detector errors; BP scales to general codes but **stumbles on degenerate codes like this one** — a real, open problem that modern research (neural/GNN decoders, BP+OSD) exists to fix.
- **No decoder beats the code's distance.** When the real error is big enough, *every* decoder fails. That's not a decoder bug — it's the threshold, and it's the whole subject of what's next.

That last point is the cliffhanger:

- **→ Notebook 3 — Noise & Thresholds** turns the error rate into a dial: sample many random errors, decode each, measure the *logical* error rate, and watch the **threshold** appear — the crossover where making the code bigger finally starts helping. (This is where the motivational plot from Notebook 1 gets earned.)
- **→ [Module 1 · Detection (live)](https://github.com/kondshk/QEC-Explorer)** and **→ [Module 2 · Decoder Race (live)](https://github.com/kondshk/QEC-Explorer)** — replay every example here, especially **Z(1,2)**, and watch the decoders you just built race in real time.

You've built the code *and* the decoders. Next: find out exactly when they're enough.